RANDOM FOREST

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
import os
import hashlib
from sklearn.model_selection import RandomizedSearchCV

In [2]:
from google.colab import drive
# 1. Collega Drive
drive.mount('/content/drive')

# 2. Vai nella cartella dove hai i file e la cartella audio
%cd "/content/drive/MyDrive/Magistrale/Tesi/Fase2"

Mounted at /content/drive
/content/drive/MyDrive/Magistrale/Tesi/Fase2


In [3]:
SEEDS = [42, 8, 1291, 64207, 305, 91876, 12, 456, 33920, 7]

In [4]:
CSV_INPUT_PATH = f"audio_dataset.csv"
LABEL_COLUMN="portata"
LOW_HYPERPARAMETERS=False
NUMBER_OF_COMBINATIONS=50
MAIN_METRIC="MAE"
if MAIN_METRIC=="MSE":
  base_score_metric='neg_mean_squared_error'
else:
  base_score_metric='neg_mean_absolute_error'
START_FROM_SEED_INDEX=5 #0

In [5]:
# 1. CARICAMENTO E CONVERSIONE IN GPU DATAFRAME
df = pd.read_csv(CSV_INPUT_PATH, index_col=0)
# Convertiamo in float32: è vitale per le performance e compatibilità GPU
#df_gpu = cudf.from_pandas(df).astype(np.float32)

In [6]:
# 2. PREPARAZIONE X e y
target_column = 'portata'
X = df.drop(columns=[target_column])
y = df[target_column]

OTTIMIZZAZIONE IPERPARAMETRI

In [7]:
# 4. DEFINIZIONE GRIGLIA PARAMETRI

param_distributions = {}

if LOW_HYPERPARAMETERS:
  param_distributions = {
      'n_estimators': [5, 8, 10, 12, 15],
      'max_depth': [3, 4, 5],
      'max_features': ['sqrt', "log2", 0.2, 0.5, 0.8, 1.0],
      'min_samples_leaf': [2, 4, 6],
      'min_samples_split': [5, 10, 15]
  }
else:
  param_distributions = {
      'n_estimators': [8, 12, 15, 20, 25],
      'max_depth': [5, 8, 10],
      'max_features': ['sqrt', "log2", 0.2, 0.5, 0.8, 1.0],
      'min_samples_leaf': [2, 4, 6],
      'min_samples_split': [5, 10, 15]
  }

In [8]:
for i, seed in enumerate(SEEDS[START_FROM_SEED_INDEX:], start=START_FROM_SEED_INDEX):

  if START_FROM_SEED_INDEX != 0:
      print(f"Skipping the first {START_FROM_SEED_INDEX} seeds...")

  print(f"\n\n=== INIZIO RANDOMIZED SEARCH CON SEME {seed} ({i+1}/{len(SEEDS)}) ===\n")

  # 3. SPLIT TRAIN/TEST
  # Dividiamo i dati: 80% training, 20% test
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

  train_indices = df.index.get_indexer(X_train.index).tolist()
  test_indices = df.index.get_indexer(X_test.index).tolist()
  print("SHA256 Train set: "+hashlib.sha256(str(train_indices).encode('utf-8')).hexdigest())
  print("SHA256 Test set: "+hashlib.sha256(str(test_indices).encode('utf-8')).hexdigest())

  # 5. MODELLO
  rf_to_optimize = RandomForestRegressor(random_state=seed)

  # 6. CONFIGURAZIONE RICERCA
  randomized_search = RandomizedSearchCV(
      estimator=rf_to_optimize,
      param_distributions=param_distributions,
      cv=3,
      n_iter=NUMBER_OF_COMBINATIONS,
      refit=False,
      scoring=base_score_metric,
      verbose=2,
      random_state=42,
  )

  # 7. ESECUZIONE
  randomized_search.fit(X_train, y_train)

  print(f"Randomized search completato per il seme {seed}.")
  # Risultati
  print("Migliori parametri individuati:")
  print(randomized_search.best_params_)
  results = pd.DataFrame(randomized_search.cv_results_)
  results = results.sort_values(by="rank_test_score", ascending=True)
  RESULTS_CSV_PATH=f"Results_randomized_search_randomforest_{seed}.csv"
  results.to_csv(RESULTS_CSV_PATH, index=False)


Skipping the first 5 seeds...


=== INIZIO RANDOMIZED SEARCH CON SEME 91876 (6/10) ===

SHA256 Train set: da546bfa3caa1a1fab41fe80eed89539f992c14c4d61691e6e71d0c0492f26aa
SHA256 Test set: df90697eff376256e6ded58980ca6dab075472342b02d62f86747e341d96ce6d
Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END max_depth=5, max_features=1.0, min_samples_leaf=4, min_samples_split=10, n_estimators=15; total time=  56.9s
[CV] END max_depth=5, max_features=1.0, min_samples_leaf=4, min_samples_split=10, n_estimators=15; total time=  57.0s
[CV] END max_depth=5, max_features=1.0, min_samples_leaf=4, min_samples_split=10, n_estimators=15; total time=  56.0s
[CV] END max_depth=10, max_features=0.2, min_samples_leaf=6, min_samples_split=15, n_estimators=8; total time=  10.1s
[CV] END max_depth=10, max_features=0.2, min_samples_leaf=6, min_samples_split=15, n_estimators=8; total time=  10.8s
[CV] END max_depth=10, max_features=0.2, min_samples_leaf=6, min_samples_split=15, n_estimators

DEFINIZIONE FINALE DEL MODELLO

In [9]:
'''
best_params = randomized_search.best_params_
rf_regressor = RandomForestRegressor(**best_params, random_state=SEED)

# 4. Addestramento finale sul set completo
rf_regressor.fit(X_train, y_train)
print("Addestramento finale completato.")
'''

'\nbest_params = randomized_search.best_params_\nrf_regressor = RandomForestRegressor(**best_params, random_state=SEED)\n\n# 4. Addestramento finale sul set completo\nrf_regressor.fit(X_train, y_train)\nprint("Addestramento finale completato.")\n'

VALUTAZIONE DEL MODELLO FINALE

In [10]:
'''
# 5. PREDIZIONE E VALUTAZIONE
y_pred = rf_regressor.predict(X_test)
'''

'\n# 5. PREDIZIONE E VALUTAZIONE\ny_pred = rf_regressor.predict(X_test)\n'

In [11]:
'''
# Metriche di Regressione
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
mse=mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"--- PERFORMANCE RANDOM FOREST ---")
print(f"R^2 Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE: {mape:.2f}%")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
'''

'\n# Metriche di Regressione\nmae = mean_absolute_error(y_test, y_pred)\nmape = mean_absolute_percentage_error(y_test, y_pred) * 100\nmse=mean_squared_error(y_test, y_pred)\nrmse = np.sqrt(mse)\nr2 = r2_score(y_test, y_pred)\n\nprint(f"--- PERFORMANCE RANDOM FOREST ---")\nprint(f"R^2 Score: {r2:.4f}")\nprint(f"MAE: {mae:.4f}")\nprint(f"MAPE: {mape:.2f}%")\nprint(f"MSE: {mse:.4f}")\nprint(f"RMSE: {rmse:.4f}")\n'

In [12]:
'''
# VISUALIZZAZIONE: FEATURE IMPORTANCE
plt.figure(figsize=(10, 6))
importances = pd.Series(rf_regressor.feature_importances_, index=X.columns)
importances.nlargest(10).sort_values(ascending=True).plot(kind='barh', color='orange')
plt.title("Top 10 Feature per la stima della Portata")
plt.xlabel("Importanza Relativa")
plt.tight_layout()
plt.show()
'''

'\n# VISUALIZZAZIONE: FEATURE IMPORTANCE\nplt.figure(figsize=(10, 6))\nimportances = pd.Series(rf_regressor.feature_importances_, index=X.columns)\nimportances.nlargest(10).sort_values(ascending=True).plot(kind=\'barh\', color=\'orange\')\nplt.title("Top 10 Feature per la stima della Portata")\nplt.xlabel("Importanza Relativa")\nplt.tight_layout()\nplt.show()\n'